<a href="https://colab.research.google.com/github/shinnew9/CSE498_AI-Healthcare-Robotics/blob/main/Lab1_ObjectDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# setting up a GPU

!nvidia-smi

Sun Sep  6 03:45:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# I mounted the google drive to access to my drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pwd

/content


In [ ]:
%cd /content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics

/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics


### step 1

I will git clone in this space

In [ ]:
# !git clone https://github.com/valeriouberti/webcam-object-recognition.git

In [ ]:
# %pip install "ultralytics<=8.3.40" supervision roboflow

In [ ]:
import os
from ultralytics import YOLO

# Base directory for this assignment
base_dir = "/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics"

# Subdirectories built from base_dir
webcam_images_dir = os.path.join(base_dir, "webcam_images")
results_dir = os.path.join(base_dir, "results")

HOME = base_dir
os.chdir(HOME)
print("Current directory:", os.getcwd())

Current directory: /content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics


In [ ]:
def get_sample_images(folder, num_images=5):
    """Return the first num_images image file paths from a folder."""
    valid_ext = (".jpg", ".jpeg", ".png")
    all_images = sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(valid_ext)
    )
    return all_images[:num_images]


def run_inference(model_path, image_paths, save_name):
    """Run YOLO inference on given images and save annotated results under results_dir/save_name."""
    model = YOLO(model_path)
    results = model.predict(image_paths, conf=0.25, save=True, project=results_dir, name=save_name)
    for r in results:
        r.show()
    return results

In [ ]:
# Pick 5 sample images from my 50 webcam images
test_images = get_sample_images(webcam_images_dir, num_images=5)
print("Selected test images:", test_images)

# Step 1: baseline test with the pretrained (not fine-tuned) model
baseline_results = run_inference("yolo11n.pt", test_images, save_name="baseline")

In [ ]:
# from ultralytics import YOLO

# model = YOLO("webcam-object-recognition/models/yolo11n.pt")

# # trying on few items
# result1 = model.predict("./webcam_images/oliveoil.jpg", conf=0.5)
# result1[0].show()

In [ ]:
# result2 = model.predict("./webcam_images/pencilholder.jpg", conf=0.5)
# result2[0].show()

In [ ]:
# result3 = model.predict("./webcam_images/kitchentowel.jpg", conf=0.5)
# result3[0].show()

### Step 2 - Objection detection on custom dataset
following this [link](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolo11-object-detection-on-custom-dataset.ipynb
)

In [ ]:
# 1st, I will download the dataset from kaggle

!mkdir -p data
import kagglehub
path = kagglehub.dataset_download("elvinrustam/grocery-dataset")
print(path)

Using Colab cache for faster access to the 'grocery-dataset' dataset.
/kaggle/input/grocery-dataset


In [ ]:
import shutil, os

dest_path = "//content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics/grocery-dataset"

# using the same paths
shutil.copytree(path, dest_path, dirs_exist_ok=True)

print("Copied files:", os.listdir(dest_path))
print(os.listdir(dest_path))

Copied files: ['GroceryDataset.csv']
['GroceryDataset.csv']


In [ ]:
import pandas as pd

df = pd.read_csv(f"{dest_path}/GroceryDataset.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (1757, 8)
Columns: ['Sub Category', 'Price', 'Discount', 'Rating', 'Title', 'Currency', 'Feature', 'Product Description']


,Sub Category,Price,Discount,Rating,Title,Currency,Feature,Product Description
0,Bakery & Desserts,$56.99,No Discount,Rated 4.3 out of 5 stars based on 265 reviews.,"David’s Cookies Mile High Peanut Butter Cake, ...",$,"""10"""" Peanut Butter Cake\nCertified Kosher OU-...",A cake the dessert epicure will die for!Our To...
1,Bakery & Desserts,$159.99,No Discount,Rated 5 out of 5 stars based on 1 reviews.,"The Cake Bake Shop 8"" Round Carrot Cake (16-22...",$,Spiced Carrot Cake with Cream Cheese Frosting ...,"Due to the perishable nature of this item, ord..."
2,Bakery & Desserts,$44.99,No Discount,Rated 4.1 out of 5 stars based on 441 reviews.,"St Michel Madeleine, Classic French Sponge Cak...",$,100 count\nIndividually wrapped\nMade in and I...,Moist and buttery sponge cakes with the tradit...
3,Bakery & Desserts,$39.99,No Discount,Rated 4.7 out of 5 stars based on 9459 reviews.,"David's Cookies Butter Pecan Meltaways 32 oz, ...",$,Butter Pecan Meltaways\n32 oz 2-Pack\nNo Prese...,These delectable butter pecan meltaways are th...
4,Bakery & Desserts,$59.99,No Discount,Rated 4.5 out of 5 stars based on 758 reviews.,"David’s Cookies Premier Chocolate Cake, 7.2 lb...",$,"""10"" Four Layer Chocolate Cake\nCertified Kosh...",A cake the dessert epicure will die for!To the...


In [ ]:
# Following the instruction on the link from cell after working on RoboFlow

from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("yoojin-shin")
project = workspace.project("lab1-uopop")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo11s.pt data={dataset.location}/data.yaml epochs=10 imgsz=640 plots=True

/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics
New https://pypi.org/project/ultralytics/8.4.142 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolo11s.pt, data=/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics/Lab1-1/data.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visua

In [ ]:
import os

save_dir = os.path.join(HOME, "runs", "detect", "train2")
weights_path = os.path.join(save_dir, "weights", "best.pt")
print(weights_path)

print(os.path.exists(weights_path))  # True가 나와야 함

/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics/runs/detect/train2/weights/best.pt
True


In [ ]:
finetuned_results = run_inference(weights_path, test_images, save_name="after_finetune")

In [ ]:
from ultralytics import YOLO

model = YOLO(weights_path)  # load the fine-tuned model (Using the same weights_path, that I have made in cell 25)

# Lower the confidence threshold to see if there are weak (low-confidence) detections
low_conf_results = model.predict(test_images, conf=0.01)
for r in low_conf_results:
    print(r.boxes.cls, r.boxes.conf)


0: 640x640 9 energydrinks, 4 frypans, 4 granolabars, 1 hanger, 1 mealkit, 1 oliveoil, 1 pinksalt, 2 slipperss, 29 spams, 8 sunglassess, 12.9ms
1: 640x640 1 crocs, 3 deskstopwatchs, 4 energydrinks, 1 frypan, 4 granolabars, 4 mintgums, 2 oliveoils, 1 pencilcase, 1 pinksalt, 1 plate, 20 spams, 4 wetwipess, 12.9ms
2: 640x640 4 bananass, 8 energydrinks, 3 frypans, 8 oliveoils, 1 slippers, 8 spams, 1 sunglasses, 12.9ms
3: 640x640 2 energydrinks, 3 granolabars, 1 journalnote, 1 minijam, 9 mintgums, 2 oliveoils, 2 pencilcases, 12 pinksalts, 29 spams, 12.9ms
4: 640x640 2 energydrinks, 1 hairpin, 1 mealkit, 1 mintgum, 3 oliveoils, 1 spam, 1 vrheadset, 12.9ms
Speed: 6.1ms preprocess, 12.9ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)
tensor([40., 14., 42., 10., 38., 22., 11., 10., 38.,  7.,  7., 40.,  7., 40., 11., 40., 40., 40., 42., 33., 40., 40., 40., 40., 40., 40., 40., 40., 10., 40., 40., 11., 40., 42., 40., 40., 40., 40., 42.,  7.,  7., 40., 42.,  7., 11., 40., 40., 40

### Exp 2 following this [link](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/zero-shot-object-detection-and-segmentation-with-yoloe.ipynb)